In [1]:
"""
Project Medusa: A Multi-Agent Legal Assistant for Military Justice (UCMJ/MCM)
============================================================================

A runnable reference implementation of the agentic architecture described in
the Project Medusa research document. It is dependency-free (pure Python stdlib)
so it can be executed directly, while clearly marking where production systems
would swap in real services (Claude, LegalBERT, AWS KMS, Neo4j, a vector DB,
Microsoft Presidio, etc.).

Architecture map -> code:
  Sec 2.1  Foundational LLM/embedding interfaces .... ClaudeClient / LegalBERTEmbedder
  Sec 2.2  Hybrid KG + Vector knowledge base ........ KnowledgeGraph / VectorStore / KnowledgeBase
  Sec 3.1  Hybrid orchestrator (router/supervisor/swarm) ... Orchestrator + Router + Blackboard
  Sec 3.2  Specialized agents ....................... Agent subclasses
  Sec 4    Reversible PII pseudonymization + envelope encryption ... PseudonymizationPipeline / KMS
  Sec 5.1  GraphRAG + Legal Syllogism Prompting ..... GraphRAG / SyllogismReasoner
  Sec 5.2  RLHF feedback loop ....................... FeedbackStore / RewardModel
  Sec 6.1  Provenance ledger (hash-chained, WORM) ... ProvenanceLedger
  Sec 6.2  Multi-layered guardrails ................. GuardrailStack

Run:  python project_medusa.py
"""

from __future__ import annotations

import hashlib
import hmac
import json
import os
import re
import secrets
import time
import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any, Callable, Dict, List, Optional, Tuple


# =============================================================================
# Utility: hashing & timestamps (Sec 6.1 cryptographic hashing)
# =============================================================================

def sha256(data: Any) -> str:
    """SHA-256 over a JSON-canonicalized representation of any artifact."""
    if not isinstance(data, (str, bytes)):
        data = json.dumps(data, sort_keys=True, default=str)
    if isinstance(data, str):
        data = data.encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def now_ms() -> int:
    return int(time.time() * 1000)


def new_id(prefix: str = "id") -> str:
    return f"{prefix}-{uuid.uuid4().hex[:12]}"


# =============================================================================
# Sec 6.1 -- Provenance Ledger (immutable, hash-chained, WORM-style)
# =============================================================================

@dataclass(frozen=True)
class ProvenanceEvent:
    event_id: str
    correlation_id: str       # links every event of one query (Sec 6.1)
    actor: str                # WHO
    action: str               # WHAT
    timestamp_ms: int         # WHEN
    location: str             # WHERE (component)
    rationale: str            # WHY
    outcome_hash: str         # OUTCOME (hash of the produced artifact)
    payload_hash: str         # hash of the input artifact
    prev_hash: str            # chain link to previous ledger entry

    def entry_hash(self) -> str:
        return sha256(asdict(self))


class ProvenanceLedger:
    """Append-only, hash-chained ledger emulating a WORM/blockchain store."""

    def __init__(self) -> None:
        self._chain: List[ProvenanceEvent] = []

    @property
    def tip(self) -> str:
        return self._chain[-1].entry_hash() if self._chain else "GENESIS"

    def record(self, correlation_id: str, actor: str, action: str,
               location: str, rationale: str,
               input_artifact: Any = None, output_artifact: Any = None) -> ProvenanceEvent:
        ev = ProvenanceEvent(
            event_id=new_id("evt"),
            correlation_id=correlation_id,
            actor=actor,
            action=action,
            timestamp_ms=now_ms(),
            location=location,
            rationale=rationale,
            outcome_hash=sha256(output_artifact) if output_artifact is not None else "",
            payload_hash=sha256(input_artifact) if input_artifact is not None else "",
            prev_hash=self.tip,
        )
        self._chain.append(ev)
        return ev

    def verify_integrity(self) -> bool:
        """Validate the hash chain (Merkle-style verifiable chain of evidence)."""
        prev = "GENESIS"
        for ev in self._chain:
            if ev.prev_hash != prev:
                return False
            prev = ev.entry_hash()
        return True

    def trace(self, correlation_id: str) -> List[ProvenanceEvent]:
        return [e for e in self._chain if e.correlation_id == correlation_id]


# =============================================================================
# Sec 4.4 -- Hardware-backed KMS (emulated) + Envelope Encryption
# =============================================================================

class EmulatedKMS:
    """
    Stand-in for AWS KMS / Azure Key Vault. Holds a non-exportable Customer
    Master Key (CMK) and issues per-value Data Encryption Keys (DEKs).
    Plaintext DEKs are returned only transiently and never stored (Sec 4.4).
    """

    def __init__(self) -> None:
        self._cmk = secrets.token_bytes(32)  # never leaves the KMS boundary

    def generate_data_key(self) -> Tuple[bytes, bytes]:
        """Return (plaintext_dek, encrypted_dek). Caller must discard plaintext."""
        dek = secrets.token_bytes(32)
        encrypted_dek = self._xor_wrap(dek, self._cmk)
        return dek, encrypted_dek

    def decrypt_data_key(self, encrypted_dek: bytes) -> bytes:
        return self._xor_wrap(encrypted_dek, self._cmk)

    @staticmethod
    def _xor_wrap(data: bytes, key: bytes) -> bytes:
        # NOTE: XOR is a teaching placeholder. Production = AES-GCM inside KMS.
        return bytes(b ^ key[i % len(key)] for i, b in enumerate(data))


def envelope_encrypt(plaintext: str, kms: EmulatedKMS) -> Tuple[bytes, bytes]:
    """Encrypt PII with a per-value DEK; discard the plaintext DEK immediately."""
    dek, encrypted_dek = kms.generate_data_key()
    pt = plaintext.encode("utf-8")
    ciphertext = bytes(b ^ dek[i % len(dek)] for i, b in enumerate(pt))
    del dek  # discard plaintext DEK (Sec 4.4)
    return ciphertext, encrypted_dek


def envelope_decrypt(ciphertext: bytes, encrypted_dek: bytes, kms: EmulatedKMS) -> str:
    dek = kms.decrypt_data_key(encrypted_dek)
    pt = bytes(b ^ dek[i % len(dek)] for i, b in enumerate(ciphertext))
    del dek
    return pt.decode("utf-8")


# =============================================================================
# Sec 4 -- Reversible PII Pseudonymization Pipeline
# =============================================================================

@dataclass
class PIIMapping:
    case_id: str
    token: str
    pii_type: str
    encrypted_value: bytes
    encrypted_dek: bytes


class SecureMappingStore:
    """Emulates an encrypted mapping DB (e.g., DynamoDB) with least-privilege."""

    def __init__(self) -> None:
        self._store: Dict[str, PIIMapping] = {}

    def put(self, mapping: PIIMapping) -> None:
        self._store[mapping.token] = mapping

    def get(self, token: str) -> Optional[PIIMapping]:
        return self._store.get(token)


class PseudonymizationPipeline:
    """
    Sec 4.3: Hybrid PII detection (regex + NER stub) + coreference grouping,
    tokenization, reversible mapping via envelope encryption.
    """

    # Structured-PII regex (Sec 4.3 step 1)
    PATTERNS = {
        "SSN": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
        "EMAIL": re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
        "PHONE": re.compile(r"\b\d{3}[-.]\d{3}[-.]\d{4}\b"),
        "RANK_NAME": re.compile(r"\b(?:SGT|SPC|PFC|PVT|CPT|LT|MAJ|COL|SSG)\s+[A-Z][a-z]+\b"),
        "PERSON": re.compile(r"\b(?:Mr\.|Ms\.|Mrs\.|Dr\.)\s+[A-Z][a-z]+\b"),
    }

    def __init__(self, kms: EmulatedKMS, store: SecureMappingStore) -> None:
        self.kms = kms
        self.store = store

    def pseudonymize(self, text: str, case_id: str) -> Tuple[str, List[str]]:
        """Return (pseudonymized_text, list_of_tokens). Reversible & auditable."""
        tokens_created: List[str] = []
        # coreference grouping stub: identical surface forms -> same token
        value_to_token: Dict[str, str] = {}

        for pii_type, pattern in self.PATTERNS.items():
            for match in set(pattern.findall(text)):
                if match in value_to_token:
                    token = value_to_token[match]
                else:
                    token = f"[{pii_type}-{secrets.token_hex(3).upper()}]"
                    value_to_token[match] = token
                    ct, edek = envelope_encrypt(match, self.kms)
                    self.store.put(PIIMapping(case_id, token, pii_type, ct, edek))
                    tokens_created.append(token)
                text = text.replace(match, token)
        return text, tokens_created

    # Sec 4.5: de-anonymization gated by RBAC + ABAC
    def deanonymize(self, token: str, user: "User", case_id: str) -> Optional[str]:
        if not AccessControl.can_deanonymize(user, case_id):
            raise PermissionError("RBAC/ABAC denied: requires CaseLead on this CaseID + MFA.")
        m = self.store.get(token)
        if not m or m.case_id != case_id:
            return None
        return envelope_decrypt(m.encrypted_value, m.encrypted_dek, self.kms)


# =============================================================================
# Sec 4.5 -- AuthN/AuthZ: RBAC + ABAC (emulated)
# =============================================================================

@dataclass
class User:
    user_id: str
    roles: List[str]               # RBAC
    assigned_cases: List[str]      # ABAC attribute
    mfa_verified: bool = False


class AccessControl:
    @staticmethod
    def can_deanonymize(user: User, case_id: str) -> bool:
        return (
            user.mfa_verified
            and "CaseLead" in user.roles
            and case_id in user.assigned_cases
        )


# =============================================================================
# Sec 2.1 -- LLM & Embedding interfaces (stubs for Claude / LegalBERT)
# =============================================================================

class ClaudeClient:
    """
    Stub for Anthropic Claude. In production, call the Messages API.
    Here it returns deterministic, structured text so the pipeline runs offline.
    """

    MODEL = "claude-medusa-stub-v1"

    def complete(self, prompt: str, system: str = "") -> str:
        # Deterministic stub that echoes structure for demonstration/testing.
        digest = sha256(system + prompt)[:8]
        return f"[Claude:{self.MODEL}#{digest}] " + self._mock_reasoning(prompt)

    @staticmethod
    def _mock_reasoning(prompt: str) -> str:
        if "syllogism" in prompt.lower():
            return ("MAJOR PREMISE (Rule): see retrieved UCMJ article. "
                    "MINOR PREMISE (Facts): see case facts. "
                    "CONCLUSION: application of rule to facts yields a reasoned judgment "
                    "(decision-support only).")
        return "Synthesized response grounded in retrieved knowledge."


class LegalBERTEmbedder:
    """
    Stub for LegalBERT embeddings (Sec 2.1). Produces a deterministic
    pseudo-embedding so cosine similarity is meaningful for the demo.
    """

    DIM = 64

    def embed(self, text: str) -> List[float]:
        vec = [0.0] * self.DIM
        for tok in re.findall(r"\w+", text.lower()):
            h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
            vec[h % self.DIM] += 1.0
        norm = sum(v * v for v in vec) ** 0.5 or 1.0
        return [v / norm for v in vec]


def cosine(a: List[float], b: List[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


# =============================================================================
# Sec 2.2 -- Hybrid Knowledge Base: Knowledge Graph + Vector Store
# =============================================================================

@dataclass
class GraphNode:
    node_id: str
    node_type: str        # e.g., LegalIssue, Rule, Source, Precedent (IRAC schema)
    label: str
    attrs: Dict[str, Any] = field(default_factory=dict)


@dataclass
class GraphEdge:
    src: str
    dst: str
    relation: str         # cites, defines, applies, arises_from, leads_to


class KnowledgeGraph:
    """Sec 2.2: KG for explicit logical structure (IRAC / 3-layer ontology)."""

    def __init__(self) -> None:
        self.nodes: Dict[str, GraphNode] = {}
        self.edges: List[GraphEdge] = []

    def add_node(self, node: GraphNode) -> None:
        self.nodes[node.node_id] = node

    def add_edge(self, src: str, dst: str, relation: str) -> None:
        self.edges.append(GraphEdge(src, dst, relation))

    def neighbors(self, node_id: str, relation: Optional[str] = None) -> List[GraphNode]:
        out = []
        for e in self.edges:
            if e.src == node_id and (relation is None or e.relation == relation):
                if e.dst in self.nodes:
                    out.append(self.nodes[e.dst])
        return out

    def traverse_issue_to_rule(self, issue_id: str) -> Dict[str, Any]:
        """Graph traversal: LegalIssue --applies--> Rule --cites--> Source."""
        path = {"issue": self.nodes.get(issue_id), "rules": [], "sources": []}
        for rule in self.neighbors(issue_id, "applies"):
            path["rules"].append(rule)
            path["sources"].extend(self.neighbors(rule.node_id, "cites"))
        return path


@dataclass
class VectorRecord:
    chunk_id: str
    text: str
    embedding: List[float]
    metadata: Dict[str, Any]      # Sec 2.2: e.g., UCMJ article, MCM version


class VectorStore:
    """Sec 2.2: semantic search with metadata pre-filtering."""

    def __init__(self, embedder: LegalBERTEmbedder) -> None:
        self.embedder = embedder
        self.records: List[VectorRecord] = []

    def add(self, text: str, metadata: Dict[str, Any]) -> None:
        self.records.append(
            VectorRecord(new_id("chunk"), text, self.embedder.embed(text), metadata)
        )

    def search(self, query: str, k: int = 3,
               metadata_filter: Optional[Dict[str, Any]] = None) -> List[Tuple[VectorRecord, float]]:
        q = self.embedder.embed(query)
        pool = self.records
        if metadata_filter:
            pool = [r for r in pool
                    if all(r.metadata.get(kk) == vv for kk, vv in metadata_filter.items())]
        scored = [(r, cosine(q, r.embedding)) for r in pool]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]


class KnowledgeBase:
    """Bundles KG + VectorStore (the UCMJ/MCM/case-law corpus)."""

    def __init__(self, embedder: LegalBERTEmbedder) -> None:
        self.kg = KnowledgeGraph()
        self.vectors = VectorStore(embedder)

    def seed_ucmj_demo(self) -> None:
        """Minimal UCMJ seed data demonstrating the IRAC graph schema."""
        # Reference KG: canonical UCMJ
        art121 = GraphNode("src:ucmj-121", "Source", "UCMJ Art. 121 (Larceny & Wrongful Appropriation)",
                           {"article": "121"})
        rule121 = GraphNode("rule:larceny", "Rule",
                            "Wrongful taking of property of another with intent to permanently deprive.",
                            {"article": "121"})
        issue = GraphNode("issue:theft", "LegalIssue", "Was government property wrongfully taken?")
        self.kg.add_node(art121); self.kg.add_node(rule121); self.kg.add_node(issue)
        self.kg.add_edge(issue.node_id, rule121.node_id, "applies")
        self.kg.add_edge(rule121.node_id, art121.node_id, "cites")

        # Vector store: text chunks with metadata
        self.vectors.add(
            "Under UCMJ Article 121, larceny requires a wrongful taking and intent to "
            "permanently deprive the owner of property.",
            {"article": "121", "source": "UCMJ", "mcm_version": "2024"},
        )
        self.vectors.add(
            "Wrongful appropriation differs from larceny by intent to temporarily deprive.",
            {"article": "121", "source": "UCMJ", "mcm_version": "2024"},
        )
        self.vectors.add(
            "Article 134 covers general offenses prejudicial to good order and discipline.",
            {"article": "134", "source": "UCMJ", "mcm_version": "2024"},
        )


# =============================================================================
# Sec 6.2 -- Multi-Layered Guardrail Stack
# =============================================================================

class GuardrailViolation(Exception):
    pass


class GuardrailStack:
    INJECTION = re.compile(r"(ignore previous|disregard.*instructions|system prompt)", re.I)
    PII_LEAK = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")  # raw SSN in output
    DISCLAIMER = ("\n\n[DISCLAIMER] This output is AI-generated decision-support and "
                  "does NOT constitute legal advice. A qualified judge advocate must "
                  "review and make all final decisions.")

    def __init__(self, allowed_corpora: List[str]) -> None:
        self.allowed_corpora = set(allowed_corpora)

    # Level 1: input filtering
    def check_prompt(self, prompt: str) -> None:
        if self.INJECTION.search(prompt):
            raise GuardrailViolation("L1 prompt guardrail: possible prompt injection blocked.")

    # Level 2: retrieval protection
    def check_retrieval(self, records: List[VectorRecord]) -> List[VectorRecord]:
        clean = []
        for r in records:
            if r.metadata.get("source") not in self.allowed_corpora:
                continue  # only authorized legal corpora
            sanitized = self.INJECTION.sub("[REDACTED]", r.text)  # strip embedded instructions
            clean.append(VectorRecord(r.chunk_id, sanitized, r.embedding, r.metadata))
        return clean

    # Level 3: output filtering
    def check_output(self, text: str) -> str:
        if self.PII_LEAK.search(text):
            raise GuardrailViolation("L3 output guardrail: raw PII leak detected.")
        return text + self.DISCLAIMER

    # Level 4: action governance
    @staticmethod
    def check_action(user: User, action: str, required_role: str) -> None:
        if required_role not in user.roles:
            raise GuardrailViolation(f"L4 action guardrail: '{action}' requires role '{required_role}'.")


# =============================================================================
# Sec 5.1 -- GraphRAG + Legal Syllogism Prompting
# =============================================================================

@dataclass
class RetrievalResult:
    graph_path: Dict[str, Any]
    semantic_chunks: List[VectorRecord]


class GraphRAG:
    """Sec 5.1: two-pronged retrieval (graph traversal + semantic search)."""

    def __init__(self, kb: KnowledgeBase, guardrails: GuardrailStack) -> None:
        self.kb = kb
        self.guardrails = guardrails

    def retrieve(self, query: str, issue_id: str,
                 metadata_filter: Optional[Dict[str, Any]] = None) -> RetrievalResult:
        graph_path = self.kb.kg.traverse_issue_to_rule(issue_id)
        raw = [r for r, _ in self.kb.vectors.search(query, k=4, metadata_filter=metadata_filter)]
        clean = self.guardrails.check_retrieval(raw)
        return RetrievalResult(graph_path, clean)


class SyllogismReasoner:
    """Sec 5.1: Legal Syllogism Prompting (Logic-of-Thought) over GraphRAG context."""

    SYSTEM = ("You are a UCMJ/MCM legal reasoning assistant. Use strict legal "
              "syllogism. Cite retrieved sources. Provide decision-support only.")

    def __init__(self, claude: ClaudeClient) -> None:
        self.claude = claude

    def build_prompt(self, facts: str, retrieval: RetrievalResult) -> str:
        rules = "; ".join(r.label for r in retrieval.graph_path.get("rules", [])) or "N/A"
        sources = "; ".join(s.label for s in retrieval.graph_path.get("sources", [])) or "N/A"
        precedents = " | ".join(c.text for c in retrieval.semantic_chunks) or "N/A"
        return (
            "In the legal syllogism, the major premise is the law article, the minor "
            "premise is the facts of the case, and the conclusion is the judgment.\n\n"
            f"MAJOR PREMISE (Rule from KG): {rules}\n"
            f"GOVERNING SOURCE: {sources}\n"
            f"RETRIEVED PRECEDENTS (vector): {precedents}\n\n"
            f"Case: {facts}\n\n"
            "Let us use legal syllogism to think and output the judgment (with "
            "chain-of-thought):"
        )

    def reason(self, facts: str, retrieval: RetrievalResult) -> str:
        prompt = self.build_prompt(facts, retrieval)
        return self.claude.complete(prompt, system=self.SYSTEM)


# =============================================================================
# Sec 3 -- Multi-Agent System
# =============================================================================

class AgentTask(Enum):
    EVIDENCE_COLLECTION = "evidence_collection"
    GOVERNANCE_DESIGN = "governance_design"
    GOVERNANCE_COORDINATION = "governance_coordination"
    STAKEHOLDER_INTEL = "stakeholder_intel"
    RISK_PREDICTION = "risk_prediction"
    RESOURCE_ALLOCATION = "resource_allocation"
    LEGAL_RESEARCH = "legal_research"
    LEGAL_DRAFTING = "legal_drafting"


@dataclass
class Message:
    correlation_id: str
    sender: str
    task: AgentTask
    content: Dict[str, Any]


class Blackboard:
    """Sec 3.1: shared space for the collaborative swarm / event-driven mode."""

    def __init__(self) -> None:
        self._space: Dict[str, Dict[str, Any]] = {}

    def write(self, correlation_id: str, key: str, value: Any) -> None:
        self._space.setdefault(correlation_id, {})[key] = value

    def read(self, correlation_id: str) -> Dict[str, Any]:
        return self._space.get(correlation_id, {})


class Agent(ABC):
    VERSION = "1.0.0"

    def __init__(self, name: str, ledger: ProvenanceLedger) -> None:
        self.name = name
        self.ledger = ledger

    @abstractmethod
    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        ...

    def _log(self, msg: Message, rationale: str, output: Any) -> None:
        # Sec 6.1: agent logs version, input hash, output.
        self.ledger.record(
            correlation_id=msg.correlation_id,
            actor=f"{self.name}@{self.VERSION}",
            action=f"handle:{msg.task.value}",
            location="MAS",
            rationale=rationale,
            input_artifact=msg.content,
            output_artifact=output,
        )


# ---- Sec 3.2 specialized agents -------------------------------------------

class EvidenceCollectionAgent(Agent):
    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        raw_text = msg.content["document_text"]
        case_id = msg.content["case_id"]
        # data provenance: hash of original document (Sec 6.1)
        ctx.ledger.record(msg.correlation_id, self.name, "ingest_document",
                          "EvidencePipeline", "Hashing source for provenance",
                          input_artifact=raw_text, output_artifact={"doc_hash": sha256(raw_text)})
        # Sec 4: pseudonymize before anything else touches it
        clean_text, tokens = ctx.pseudonymizer.pseudonymize(raw_text, case_id)
        out = {"pseudonymized_text": clean_text, "tokens": tokens,
               "doc_hash": sha256(raw_text)}
        self._log(msg, "Ingested + pseudonymized case document.", out)
        return out


class GovernanceDesignAgent(Agent):
    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        facts = msg.content["pseudonymized_text"]
        out = {
            "key_legal_questions": [
                "Which UCMJ articles are implicated by the alleged conduct?",
                "What are the elements the prosecution must prove?",
            ],
            "litigation_roadmap": ["Arraignment", "Motions", "Discovery",
                                   "Trial on merits", "Sentencing"],
            "candidate_issue_id": "issue:theft",
        }
        self._log(msg, "Proposed governance framework / roadmap.", out)
        return out


class GovernanceCoordinationAgent(Agent):
    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        roadmap = msg.content.get("litigation_roadmap", [])
        out = {"schedule": [{"phase": p, "deadline_days": (i + 1) * 14}
                            for i, p in enumerate(roadmap)],
               "compliance_status": "on_track"}
        self._log(msg, "Built deadline schedule + compliance tracking.", out)
        return out


class StakeholderIntelAgent(Agent):
    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        text = msg.content["pseudonymized_text"]
        # operate purely on pseudonymized tokens (Sec 3.2)
        tokens = re.findall(r"\[[A-Z_]+-[0-9A-F]+\]", text)
        edges = [(tokens[i], tokens[i + 1], "co_mentioned")
                 for i in range(len(tokens) - 1)]
        out = {"entities": list(set(tokens)), "relationships": edges}
        self._log(msg, "Mapped stakeholder network from pseudonymized tokens.", out)
        return out


class RiskPredictionAgent(Agent):
    """Sec 3.2: feature engineering + model + SHAP-style explanation."""

    def _features(self, content: Dict[str, Any]) -> Dict[str, float]:
        text = content.get("pseudonymized_text", "")
        return {
            "doc_length": min(len(text) / 1000.0, 5.0),          # structured
            "num_entities": float(len(set(re.findall(r"\[[A-Z_]+-", text)))),
            "mentions_intent": 1.0 if "intent" in text.lower() else 0.0,  # NLP-derived
            "mentions_property": 1.0 if "property" in text.lower() else 0.0,
        }

    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        f = self._features(msg.content)
        # Emulated gradient-boosting score (probability of motion success)
        weights = {"doc_length": 0.05, "num_entities": -0.04,
                   "mentions_intent": 0.30, "mentions_property": 0.25}
        logit = sum(weights[k] * v for k, v in f.items())
        prob = 1 / (1 + pow(2.718281828, -logit))
        # SHAP-style local attributions (Sec 3.2 XAI / Sec 6.1 feature importance)
        shap = {k: round(weights[k] * v, 4) for k, v in f.items()}
        out = {
            "motion_success_probability": round(prob, 3),
            "predicted_complexity": "high" if f["num_entities"] > 3 else "moderate",
            "appeal_likelihood": round(prob * 0.4, 3),
            "feature_importance_shap": shap,
            "fairness_check": {"equalized_odds_monitored": True},
            "model_card": {"model": "XGBoost-emulated", "version": self.VERSION},
        }
        self._log(msg, "Predicted risk with SHAP explanations.", out)
        return out


class ResourceAllocationAgent(Agent):
    """Sec 3.2: optimization using risk-prediction outputs as inputs."""

    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        complexity = msg.content.get("predicted_complexity", "moderate")
        prob = msg.content.get("motion_success_probability", 0.5)
        # simple constraint-style allocation heuristic
        attorneys = 3 if complexity == "high" else 1
        paralegals = 2 if complexity == "high" else 1
        recommendation = "proceed_to_trial" if prob >= 0.5 else "consider_settlement"
        out = {
            "allocation": {"attorneys": attorneys, "paralegals": paralegals},
            "priority_score": round(prob * (2 if complexity == "high" else 1), 3),
            "strategy_recommendation": recommendation,
        }
        self._log(msg, "Optimized resource allocation (LP/constraint heuristic).", out)
        return out


class LegalResearchAgent(Agent):
    """Sec 5.1: query decomposition + GraphRAG retrieval."""

    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        query = msg.content["query"]
        issue_id = msg.content.get("candidate_issue_id", "issue:theft")
        retrieval = ctx.graphrag.retrieve(query, issue_id,
                                          metadata_filter={"source": "UCMJ"})
        # log exact KG nodes/edges + chunks retrieved (Sec 6.1)
        out = {
            "issue_id": issue_id,
            "rules": [r.label for r in retrieval.graph_path.get("rules", [])],
            "sources": [s.label for s in retrieval.graph_path.get("sources", [])],
            "chunks": [c.text for c in retrieval.semantic_chunks],
        }
        ctx._last_retrieval = retrieval  # pass structured object to drafting
        self._log(msg, "GraphRAG hybrid retrieval (graph + semantic).", out)
        return out


class LegalDraftingAgent(Agent):
    """Sec 5.1: syllogistic synthesis via Claude, then output guardrails."""

    def handle(self, msg: Message, ctx: "SystemContext") -> Dict[str, Any]:
        facts = msg.content["pseudonymized_text"]
        retrieval = ctx._last_retrieval
        draft = ctx.syllogism.reason(facts, retrieval)
        # Sec 6.1: log full LLM context + CoT reasoning trace
        ctx.ledger.record(msg.correlation_id, self.name, "llm_reasoning_trace",
                          "Claude", "Captured full prompt + CoT for transparency",
                          input_artifact=ctx.syllogism.build_prompt(facts, retrieval),
                          output_artifact=draft)
        safe = ctx.guardrails.check_output(draft)
        out = {"draft": safe}
        self._log(msg, "Drafted syllogistic legal analysis (guardrailed).", out)
        return out


# =============================================================================
# Sec 5.2 -- RLHF feedback loop
# =============================================================================

class CritiqueCategory(Enum):
    INCORRECT_RULE = "Incorrect Rule Applied"
    FLAWED_LOGIC = "Flawed Logical Reasoning"
    HALLUCINATED_CITATION = "Hallucinated Citation"
    NONE = "None"


@dataclass
class Feedback:
    correlation_id: str
    target_agent: str
    scalar_ratings: Dict[str, int]          # 1-5 across axes
    preferred_response: Optional[str]        # pairwise winner id
    corrective_edit: Optional[str]
    structured_critique: CritiqueCategory


class RewardModel:
    """Sec 5.2: hybrid reward = w_acc*R_acc + w_util*R_util, with dense penalties."""

    def __init__(self, w_accuracy: float = 0.7, w_utility: float = 0.3) -> None:
        self.w_accuracy = w_accuracy
        self.w_utility = w_utility

    def reward(self, fb: Feedback) -> float:
        acc_axes = ["Legal Accuracy", "Citation Quality"]
        util_axes = ["Clarity", "Completeness"]
        r_acc = sum(fb.scalar_ratings.get(a, 3) for a in acc_axes) / (5 * len(acc_axes))
        r_util = sum(fb.scalar_ratings.get(a, 3) for a in util_axes) / (5 * len(util_axes))
        reward = self.w_accuracy * r_acc + self.w_utility * r_util
        # dense token-level penalty for structured critiques (Sec 5.2)
        if fb.structured_critique != CritiqueCategory.NONE:
            reward -= 0.3
        return round(reward, 4)


class FeedbackStore:
    def __init__(self, reward_model: RewardModel, ledger: ProvenanceLedger) -> None:
        self.reward_model = reward_model
        self.ledger = ledger
        self.history: List[Tuple[Feedback, float]] = []

    def submit(self, fb: Feedback) -> float:
        r = self.reward_model.reward(fb)
        self.history.append((fb, r))
        # PPO-style targeted update would apply reward to fb.target_agent only.
        self.ledger.record(fb.correlation_id, "RLHF", "ingest_feedback",
                           "FeedbackPortal",
                           f"Targeted reward -> {fb.target_agent} (PPO clip, no catastrophic forgetting)",
                           input_artifact=asdict_safe(fb), output_artifact={"reward": r})
        return r


def asdict_safe(fb: Feedback) -> Dict[str, Any]:
    d = asdict(fb)
    d["structured_critique"] = fb.structured_critique.value
    return d


# =============================================================================
# Sec 3.1 -- Orchestrator (Router-Solver: Supervisor + Swarm)
# =============================================================================

class CoordinationMode(Enum):
    CENTRAL_PLANNER = "central_planner"   # supervisor pattern (procedural)
    EVENT_DRIVEN = "event_driven"         # reactive swarm (ambiguous)


class Router:
    """Sec 3.1: classifies tasks -> procedural (planner) vs ambiguous (swarm)."""

    PROCEDURAL = re.compile(r"(compliance|deadline|article|elements|draft motion)", re.I)

    def classify(self, query: str) -> CoordinationMode:
        return (CoordinationMode.CENTRAL_PLANNER
                if self.PROCEDURAL.search(query)
                else CoordinationMode.EVENT_DRIVEN)


@dataclass
class SystemContext:
    ledger: ProvenanceLedger
    pseudonymizer: PseudonymizationPipeline
    graphrag: GraphRAG
    syllogism: SyllogismReasoner
    guardrails: GuardrailStack
    blackboard: Blackboard
    _last_retrieval: Optional[RetrievalResult] = None


class Orchestrator:
    """Sec 3.1: hybrid coordinator with dynamic switching."""

    def __init__(self, agents: Dict[AgentTask, Agent], ctx: SystemContext) -> None:
        self.agents = agents
        self.ctx = ctx
        self.router = Router()

    def run_case(self, query: str, document_text: str, case_id: str) -> Dict[str, Any]:
        cid = new_id("corr")
        self.ctx.guardrails.check_prompt(query)   # L1 guardrail
        mode = self.router.classify(query)
        self.ctx.ledger.record(cid, "Orchestrator", "route_query", "Router",
                               f"Classified as {mode.value}",
                               input_artifact=query, output_artifact={"mode": mode.value})

        # --- Stage A: evidence collection (always first; security-critical) ---
        ev = self._dispatch(AgentTask.EVIDENCE_COLLECTION, cid,
                            {"document_text": document_text, "case_id": case_id})
        self.ctx.blackboard.write(cid, "evidence", ev)
        clean_text = ev["pseudonymized_text"]

        # --- Stage B: governance design ---
        gov = self._dispatch(AgentTask.GOVERNANCE_DESIGN, cid,
                             {"pseudonymized_text": clean_text})
        self.ctx.blackboard.write(cid, "governance", gov)

        if mode == CoordinationMode.CENTRAL_PLANNER:
            return self._central_planner(cid, case_id, clean_text, query, gov)
        return self._event_driven_swarm(cid, case_id, clean_text, query, gov)

    # Supervisor pattern: ordered, highly observable workflow
    def _central_planner(self, cid, case_id, clean_text, query, gov) -> Dict[str, Any]:
        coord = self._dispatch(AgentTask.GOVERNANCE_COORDINATION, cid, gov)
        research = self._dispatch(AgentTask.LEGAL_RESEARCH, cid,
                                  {"query": query,
                                   "candidate_issue_id": gov["candidate_issue_id"]})
        draft = self._dispatch(AgentTask.LEGAL_DRAFTING, cid,
                               {"pseudonymized_text": clean_text})
        risk = self._dispatch(AgentTask.RISK_PREDICTION, cid,
                              {"pseudonymized_text": clean_text})
        alloc = self._dispatch(AgentTask.RESOURCE_ALLOCATION, cid, risk)
        return self._assemble(cid, case_id, mode="central_planner",
                              governance=gov, coordination=coord, research=research,
                              draft=draft, risk=risk, allocation=alloc)

    # Reactive swarm: agents contribute via blackboard (order-independent)
    def _event_driven_swarm(self, cid, case_id, clean_text, query, gov) -> Dict[str, Any]:
        intel = self._dispatch(AgentTask.STAKEHOLDER_INTEL, cid,
                               {"pseudonymized_text": clean_text})
        risk = self._dispatch(AgentTask.RISK_PREDICTION, cid,
                              {"pseudonymized_text": clean_text})
        research = self._dispatch(AgentTask.LEGAL_RESEARCH, cid,
                                  {"query": query,
                                   "candidate_issue_id": gov["candidate_issue_id"]})
        draft = self._dispatch(AgentTask.LEGAL_DRAFTING, cid,
                               {"pseudonymized_text": clean_text})
        alloc = self._dispatch(AgentTask.RESOURCE_ALLOCATION, cid, risk)
        return self._assemble(cid, case_id, mode="event_driven",
                              stakeholder_intel=intel, research=research,
                              draft=draft, risk=risk, allocation=alloc)

    def _dispatch(self, task: AgentTask, cid: str, content: Dict[str, Any]) -> Dict[str, Any]:
        msg = Message(cid, "Orchestrator", task, content)
        return self.agents[task].handle(msg, self.ctx)

    def _assemble(self, cid: str, case_id: str, mode: str, **parts) -> Dict[str, Any]:
        result = {"correlation_id": cid, "case_id": case_id,
                  "coordination_mode": mode, **parts}
        self.ctx.ledger.record(cid, "Orchestrator", "assemble_response", "MAS",
                               "Synthesized final user-facing response",
                               output_artifact={"keys": list(parts.keys())})
        return result


# =============================================================================
# Demo / wiring
# =============================================================================

def build_system() -> Tuple[Orchestrator, SystemContext, FeedbackStore]:
    ledger = ProvenanceLedger()
    kms = EmulatedKMS()
    mapping_store = SecureMappingStore()
    pseudonymizer = PseudonymizationPipeline(kms, mapping_store)

    embedder = LegalBERTEmbedder()
    kb = KnowledgeBase(embedder)
    kb.seed_ucmj_demo()

    guardrails = GuardrailStack(allowed_corpora=["UCMJ", "MCM"])
    graphrag = GraphRAG(kb, guardrails)
    claude = ClaudeClient()
    syllogism = SyllogismReasoner(claude)
    blackboard = Blackboard()

    ctx = SystemContext(ledger, pseudonymizer, graphrag, syllogism,
                        guardrails, blackboard)

    agents: Dict[AgentTask, Agent] = {
        AgentTask.EVIDENCE_COLLECTION: EvidenceCollectionAgent("EvidenceCollectionAgent", ledger),
        AgentTask.GOVERNANCE_DESIGN: GovernanceDesignAgent("GovernanceDesignAgent", ledger),
        AgentTask.GOVERNANCE_COORDINATION: GovernanceCoordinationAgent("GovernanceCoordinationAgent", ledger),
        AgentTask.STAKEHOLDER_INTEL: StakeholderIntelAgent("StakeholderIntelAgent", ledger),
        AgentTask.RISK_PREDICTION: RiskPredictionAgent("RiskPredictionAgent", ledger),
        AgentTask.RESOURCE_ALLOCATION: ResourceAllocationAgent("ResourceAllocationAgent", ledger),
        AgentTask.LEGAL_RESEARCH: LegalResearchAgent("LegalResearchAgent", ledger),
        AgentTask.LEGAL_DRAFTING: LegalDraftingAgent("LegalDraftingAgent", ledger),
    }

    orchestrator = Orchestrator(agents, ctx)
    feedback = FeedbackStore(RewardModel(), ledger)
    return orchestrator, ctx, feedback


def demo() -> None:
    orchestrator, ctx, feedback = build_system()

    case_id = "CASE-2024-0042"
    document = (
        "SGT Miller is accused under UCMJ Article 121 of wrongfully taking government "
        "property with the intent to permanently deprive the unit of equipment. "
        "Mr. Davis (witness) stated he saw the property removed. "
        "Contact: john.davis@example.mil, 555-123-4567, SSN 123-45-6789."
    )
    query = "Analyze Article 121 elements and assess motion-to-dismiss success."

    print("=" * 70)
    print("PROJECT MEDUSA — Multi-Agent Legal Assistant (UCMJ/MCM)")
    print("=" * 70)

    result = orchestrator.run_case(query, document, case_id)

    print(f"\nCoordination mode: {result['coordination_mode']}")
    print(f"Correlation ID:    {result['correlation_id']}")
    print(f"\nPseudonymized text:\n  {result['governance'] and ''}"
          f"{ctx.blackboard.read(result['correlation_id'])['evidence']['pseudonymized_text']}")
    print(f"\nResearch (GraphRAG):\n  Rules:   {result['research']['rules']}")
    print(f"  Sources: {result['research']['sources']}")
    print(f"\nRisk prediction:\n  P(motion success) = {result['risk']['motion_success_probability']}")
    print(f"  SHAP attributions = {result['risk']['feature_importance_shap']}")
    print(f"\nResource allocation: {result['allocation']['allocation']} "
          f"-> {result['allocation']['strategy_recommendation']}")
    print(f"\nSyllogistic draft:\n  {result['draft']['draft'][:200]}...")

    # --- De-anonymization demo (RBAC + ABAC + MFA) ---
    print("\n" + "-" * 70)
    print("DE-ANONYMIZATION (Sec 4.5)")
    tokens = ctx.blackboard.read(result['correlation_id'])['evidence']['tokens']
    lead = User("u-lead", roles=["CaseLead"], assigned_cases=[case_id], mfa_verified=True)
    intruder = User("u-bad", roles=["Paralegal"], assigned_cases=[], mfa_verified=False)
    if tokens:
        tok = tokens[0]
        print(f"  Authorized CaseLead reveals {tok} -> "
              f"{ctx.pseudonymizer.deanonymize(tok, lead, case_id)}")
        try:
            ctx.pseudonymizer.deanonymize(tok, intruder, case_id)
        except PermissionError as e:
            print(f"  Unauthorized request blocked -> {e}")

    # --- RLHF feedback demo (Sec 5.2) ---
    print("\n" + "-" * 70)
    print("RLHF FEEDBACK (Sec 5.2)")
    fb = Feedback(
        correlation_id=result["correlation_id"],
        target_agent="LegalDraftingAgent",
        scalar_ratings={"Legal Accuracy": 4, "Citation Quality": 5,
                        "Clarity": 4, "Completeness": 3},
        preferred_response="response_A",
        corrective_edit=None,
        structured_critique=CritiqueCategory.NONE,
    )
    print(f"  Computed reward signal -> {feedback.submit(fb)}")

    # --- Provenance audit (Sec 6.1) ---
    print("\n" + "-" * 70)
    print("PROVENANCE AUDIT TRAIL (Sec 6.1)")
    print(f"  Ledger integrity valid: {ctx.ledger.verify_integrity()}")
    trail = ctx.ledger.trace(result["correlation_id"])
    print(f"  {len(trail)} events for this query:")
    for ev in trail:
        print(f"    [{ev.timestamp_ms}] {ev.actor:42s} {ev.action}")


if __name__ == "__main__":
    demo()

PROJECT MEDUSA — Multi-Agent Legal Assistant (UCMJ/MCM)

Coordination mode: central_planner
Correlation ID:    corr-1d69ad14a0b1

Pseudonymized text:
  [RANK_NAME-0212C8] is accused under UCMJ Article 121 of wrongfully taking government property with the intent to permanently deprive the unit of equipment. [PERSON-B244FF] (witness) stated he saw the property removed. Contact: [EMAIL-7B6BA4], [PHONE-55ABD1], SSN [SSN-B6983E].

Research (GraphRAG):
  Rules:   ['Wrongful taking of property of another with intent to permanently deprive.']
  Sources: ['UCMJ Art. 121 (Larceny & Wrongful Appropriation)']

Risk prediction:
  P(motion success) = 0.59
  SHAP attributions = {'doc_length': 0.0138, 'num_entities': -0.2, 'mentions_intent': 0.3, 'mentions_property': 0.25}

Resource allocation: {'attorneys': 3, 'paralegals': 2} -> proceed_to_trial

Syllogistic draft:
  [Claude:claude-medusa-stub-v1#52626e1b] MAJOR PREMISE (Rule): see retrieved UCMJ article. MINOR PREMISE (Facts): see case facts. CONCL